In [1]:
# Packages
import numpy as np
import pandas as pd
import glob

In [2]:
#The Ownership dataset (2021-2023)
df = pd.read_csv("/dsa/groups/casestudycf25/team02/OWNRSHP_PGYR2021_2023.csv")
df.head(5)

,Change_Type,Physician_Profile_ID,Physician_NPI,Physician_First_Name,Physician_Middle_Name,Physician_Last_Name,Physician_Name_Suffix,Recipient_Primary_Business_Street_Address_Line1,Recipient_Primary_Business_Street_Address_Line2,Recipient_City,...,Value_of_Interest,Terms_of_Interest,Submitting_Applicable_Manufacturer_or_Applicable_GPO_Name,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country,Dispute_Status_for_Publication,Interest_Held_by_Physician_or_an_Immediate_Family_Member,Payment_Publication_Date
0,UNCHANGED,361157,1.962476e+09,Marlene,NaN,Moster,NaN,100 Presidential Bvd,Suite 200,Bala Cynwyd,...,296.69,Equity Distribution payment,"Mobius Therapeutics, LLC",100000011188,"Mobius Therapeutics, LLC",MO,United States,No,Physician Covered Recipient,6/30/2025
1,UNCHANGED,173155,1.821079e+09,Robert,NaN,Ritch,NaN,455 W 57th Street,NaN,New York,...,593.37,Equity Distribution payment,"Mobius Therapeutics, LLC",100000011188,"Mobius Therapeutics, LLC",MO,United States,No,Physician Covered Recipient,6/30/2025
2,NEW,84927,1.881709e+09,andreluiz,NaN,davila,NaN,185 pilgrim road,baker4,Boston,...,31926.00,private equity,"Manual Surgical Sciences, LLC",100000806858,"Manual Surgical Sciences, LLC",UT,United States,No,Physician Covered Recipient,6/30/2025
3,NEW,119199,1.639176e+09,shephal,NaN,doshi,NaN,2001 santa monica blvd,ste280w,santa monica,...,43161.00,private equity,"Manual Surgical Sciences, LLC",100000806858,"Manual Surgical Sciences, LLC",UT,United States,No,Physician Covered Recipient,6/30/2025
4,NEW,233626,1.144259e+09,srinivas,NaN,dukkipati,NaN,1 gustav l. levy place,NaN,new york,...,87984.00,private equity,"Manual Surgical Sciences, LLC",100000806858,"Manual Surgical Sciences, LLC",UT,United States,No,Physician Covered Recipient,6/30/2025


In [3]:
df["Record_ID"].head(5)

0    754966674
1    754966682
2    755268702
3    755300384
4    755306684
Name: Record_ID, dtype: int64

In [4]:
import os

path = "/dsa/groups/casestudycf25/team02"

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
csv_files

['Taxonomy Code List Dec 2023.csv', 'OWNRSHP_PGYR2021_2023.csv']

In [5]:
#Shape and Columns of Ownership Datasets
print(df.shape)
print(df.columns)
df.info()

(12710, 30)
Index(['Change_Type', 'Physician_Profile_ID', 'Physician_NPI',
       'Physician_First_Name', 'Physician_Middle_Name', 'Physician_Last_Name',
       'Physician_Name_Suffix',
       'Recipient_Primary_Business_Street_Address_Line1',
       'Recipient_Primary_Business_Street_Address_Line2', 'Recipient_City',
       'Recipient_State', 'Recipient_Zip_Code', 'Recipient_Country',
       'Recipient_Province', 'Recipient_Postal_Code', 'Physician_Primary_Type',
       'Physician_Specialty', 'Record_ID', 'Program_Year',
       'Total_Amount_Invested_USDollars', 'Value_of_Interest',
       'Terms_of_Interest',
       'Submitting_Applicable_Manufacturer_or_Applicable_GPO_Name',
       'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID',
       'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name',
       'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State',
       'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country',
       'Dispute_Status_

In [6]:
df.isna().sum().sort_values(ascending=False)

Recipient_Postal_Code                                               12709
Recipient_Province                                                  12709
Physician_Name_Suffix                                               12115
Physician_Middle_Name                                                8428
Recipient_Primary_Business_Street_Address_Line2                      7033
Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State        123
Terms_of_Interest                                                      25
Physician_NPI                                                           8
Recipient_State                                                         1
Recipient_Zip_Code                                                      1
Physician_Profile_ID                                                    0
Physician_First_Name                                                    0
Physician_Last_Name                                                     0
Recipient_Primary_Business_Street_Addr

In [7]:
#Fixing the NPI formatting
# Convert Ownership NPI to string without scientific notation
df["Physician_NPI"] = df["Physician_NPI"].astype("Int64").astype(str)

#Standardize Names for Matching
df["FULL_NAME"] = (
    df["Physician_First_Name"].str.strip().str.upper() + " " +
    df["Physician_Last_Name"].str.strip().str.upper()
)

In [8]:
# Remove numbers after the dash in the zipcode column. Example 37212-4354 -> 37212
df["Recipient_Zip_Code"] = df["Recipient_Zip_Code"].str.replace(r"-\d{4}$", "", regex=True)

In [9]:
# Any column that contains 'Name' is trimmed and uppercased
print(df['FULL_NAME'].head(1))
cols = [c for c in df.columns if "name" in c.lower()]

df[cols] = df[cols].apply(lambda col: col.str.upper().str.strip())
df['FULL_NAME'].head(1)

0    MARLENE MOSTER
Name: FULL_NAME, dtype: object


0    MARLENE MOSTER
Name: FULL_NAME, dtype: object

In [10]:
# Terms_of_Interest is trimmed and uppercased
print(df['Terms_of_Interest'].head(1))

df["Terms_of_Interest"] = df["Terms_of_Interest"].str.upper().str.strip()
df['Terms_of_Interest'].head(1)

0    Equity Distribution payment
Name: Terms_of_Interest, dtype: object


0    EQUITY DISTRIBUTION PAYMENT
Name: Terms_of_Interest, dtype: object

In [11]:
# Check blank rows in manufacturer name
blank_rows = df[df["FULL_NAME"]
                  .astype(str).str.strip().eq("")]
blank_rows

,Change_Type,Physician_Profile_ID,Physician_NPI,Physician_First_Name,Physician_Middle_Name,Physician_Last_Name,Physician_Name_Suffix,Recipient_Primary_Business_Street_Address_Line1,Recipient_Primary_Business_Street_Address_Line2,Recipient_City,...,Terms_of_Interest,Submitting_Applicable_Manufacturer_or_Applicable_GPO_Name,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country,Dispute_Status_for_Publication,Interest_Held_by_Physician_or_an_Immediate_Family_Member,Payment_Publication_Date,FULL_NAME


In [12]:
###################
# Drop blank columns
#####################
df.dropna(axis=1, how="all", inplace=True)

In [13]:
###################################
# filter dataset to remove Nas in Physician_NPI
###################################
df = df.dropna(subset=["Physician_NPI"])

In [14]:
df['Dispute_Status_for_Publication'].value_counts()

No     12708
Yes        2
Name: Dispute_Status_for_Publication, dtype: int64

In [15]:
###################################
# filter dataset to remove disputed ownership records
###################################

df_filtered = df[df["Dispute_Status_for_Publication"] == "No"]

In [16]:
#####################
# Drop unrelated columns
#####################
drop_cols = [
    
    # unnecessary because it is in other dataset
    'Physician_First_Name', 'Physician_Middle_Name', 'Physician_Last_Name',
    'Physician_Name_Suffix',
    'Recipient_Primary_Business_Street_Address_Line1',
    'Recipient_Primary_Business_Street_Address_Line2', 'Recipient_City',
    'Recipient_State', 'Recipient_Zip_Code', 'Recipient_Country',
    'Recipient_Province', 'Recipient_Postal_Code', 'Physician_Primary_Type',
    'Physician_Specialty',   
    
    # unnecessary because it is in other dataset
    'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country'  ,    
    'Submitting_Applicable_Manufacturer_or_Applicable_GPO_Name'  # using application name instead
]

df = df.drop(columns=drop_cols)

In [17]:
print(f"New Dataset Shape After Dropping Cols: {df.shape}")

New Dataset Shape After Dropping Cols: (12710, 15)


In [18]:
###################################
# Check for any duplicate record
###################################
print("Check if there are any duplicate record ids in final dataset: ")
df["Record_ID"].duplicated().any()

Check if there are any duplicate record ids in final dataset: 


False

In [19]:
col_totals = df.select_dtypes(include="number").sum()
print("All Column Totals")
print("-" *50)
print(col_totals.apply(lambda x: f"{x:.0f}"))

All Column Totals
--------------------------------------------------
Physician_Profile_ID                                                 6414412036
Record_ID                                                        11934425509635
Program_Year                                                           25699747
Total_Amount_Invested_USDollars                                       936165817
Value_of_Interest                                                    4111446123
Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID    1271005988744339
dtype: object


In [20]:
df.to_csv("/dsa/groups/casestudycf25/team02/ownership_payment_clean.csv", index=False)
df_filtered.shape

(12708, 31)

In [21]:
df.to_csv("ownership_payment_clean.csv", index=False)

In [22]:
path = "/dsa/groups/casestudycf25/team02"

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
csv_files

['Taxonomy Code List Dec 2023.csv',
 'OWNRSHP_PGYR2021_2023.csv',
 'ownership_payment_clean.csv']